# Feature Importance Analysis via Progressive Feature Removal

Replicates the experiment from `using_mlp_notebook`, but removes iteratively features from the input layer, and measure which one's removal is most impactful to the model's performance

**Strategy:**
1. Start with ALL available corpus_stats features (baseline)
2. Iteratively remove one random feature at a time
3. Track metrics (accuracy, precision, recall, F1) for each configuration
4. Compare results to identify which feature removal had the greatest performance impact

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from itertools import combinations

from lexos.classification.mlp_pipeline import (
    MLPPipelineConfig,
    run_mlp_authorship_pipeline,
)
from lexos.corpus.corpus_stats import CorpusStats

In [5]:
SEED = 42
rng = random.Random(SEED)
np.random.seed(SEED)

from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer

scrubber = Scrubber()
scrubber.add_pipe("lower_case")
scrubber.add_pipe("digits")
scrubber.add_pipe("punctuation")

tokenizer = Tokenizer(model="en_core_web_sm")

base = Path.cwd()
search_roots = [base] + list(base.parents)
data_dir = next(
    (root / "fed_papers" for root in search_roots if (root / "fed_papers").exists()),
    None,
)
if data_dir is None:
    raise FileNotFoundError(
        "Could not locate 'fed_papers' from the current notebook location."
    )

train_dirs = ["HAMILTON", "MADISON"]
unknown_dirs = ["DISPUTED", "COAUTHORED"]

train_files = []
train_labels = []
for author in train_dirs:
    files = sorted((data_dir / author).glob("*.txt"))
    train_files.extend(files)
    train_labels.extend([author] * len(files))

unknown_files = []
for subset in unknown_dirs:
    unknown_files.extend(sorted((data_dir / subset).glob("*.txt")))

train_texts = [
    path.read_text(encoding="utf-8", errors="ignore") for path in train_files
]
unknown_texts = [
    path.read_text(encoding="utf-8", errors="ignore") for path in unknown_files
]
train_ids = [path.name for path in train_files]
unknown_ids = [path.name for path in unknown_files]

print(f"Using data directory: {data_dir}")
print(f"Training docs: {len(train_texts)}")
print(f"Unknown docs: {len(unknown_texts)}")
print(pd.Series(train_labels).value_counts())

Using data directory: /home/mango/Lexos_Independant_Research/lexos/doc_src/docs/tutorials/classification/fed_papers
Training docs: 65
Unknown docs: 15
HAMILTON    51
MADISON     14
Name: count, dtype: int64


### 4.2 Discover available CorpusStats features

Helper helper functions to tokenize the training texts, build a temporary CorpusStats object, and extract the numeric feature names used by the experiment sweep.

In [6]:
def tokenize_texts(texts: list[str]) -> list[list[str]]:
    token_lists: list[list[str]] = []
    for text in texts:
        clean_text = scrubber.scrub(text)
        token_doc = tokenizer(clean_text)
        tokens = [token.text for token in token_doc if token.text.strip()]
        token_lists.append(tokens)

    if any(len(tokens) == 0 for tokens in token_lists):
        raise ValueError(
            "At least one document produced zero tokens after preprocessing."
        )

    return token_lists


# Function to dinamically discover the available numeric featrues in CorpusStats for the current training data
def build_corpus_stat_features(
    texts: list[str], doc_ids: list[str], min_df: int
) -> list[str]:
    token_lists = tokenize_texts(texts)
    docs_for_corpus = [
        (doc_id, doc_id, " ".join(tokens))
        for doc_id, tokens in zip(doc_ids, token_lists)
    ]
    corpus = CorpusStats(docs=docs_for_corpus, min_df=min_df)
    stats_df = corpus.doc_stats_df.reindex(doc_ids)
    numeric_stats = stats_df.select_dtypes(include=[np.number]).copy()

    if numeric_stats.empty:
        raise ValueError(
            "CorpusStats did not produce any numeric features to evaluate."
        )

    return list(numeric_stats.columns)  # returns the list of extracted features

### 4.3 Configure the baseline experiment and removal order

Define the MLP settings, compute the full CorpusStats feature list, and fixes the random removal order used for the sweep.

In [ ]:
# Control base model configurations using all available features
baseline_cfg = MLPPipelineConfig(
    seed=SEED,
    min_df=2,
    test_size=0.2,
    cv_splits=5,
    include_bigrams=True,
    use_smote=True,
    use_corpus_stats_features=True,
    corpus_stats_feature_columns=None,
    mlp_kwargs={
        "hidden_layer_sizes": (64,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 1000,  # for the purposes of this notebook, should consider decreasing this
    },
)

all_corpus_stat_features = build_corpus_stat_features(
    train_texts, train_ids, baseline_cfg.min_df
)
feature_removal_order = all_corpus_stat_features.copy()
rng.shuffle(feature_removal_order)

print(f"CorpusStats features available: {len(all_corpus_stat_features)}")
print("Random removal order:")
print(", ".join(feature_removal_order))

CorpusStats features available: 40
Random removal order:
yule_k, total_terms, hapax_legomena, propn_count, noun_count, punc_count, aux_count, hapax_dislegomena, average_word_length, det_count, average_sentence_length, adj_count, sentence_count, emotion_word_count, hapax_legomenon_rate, part_count, pron_count, verb_count, question_count, flesch_reading_ease, adverb_count, cconj_count, sconj_count, adp_count, stop_word_count, polarity, participle_count, intj_count, character_count, total_tokens, sym_count, num_count, ttr, nominal_ratio, guiraud_index, exclamation_count, vocabulary_density, subjectivity, unique_word_count, simple_nominal_ratio


### 4.4 Run the sweep and summarize results

Run the baseline experiment, removes one CorpusStats feature at a time in a random order, and compares each configuration against the baseline.

NOTE: This cell took me (Fedora 44, Dell Inspiron 14) 75 mins to run

In [ ]:
def make_config(
    selected_features: list[str] | tuple[str, ...] | None,
) -> MLPPipelineConfig:
    """Returns an MLPPipelineConfig identical to the baseline, except for the corpus_stats_feature_columns"""
    return MLPPipelineConfig(
        seed=baseline_cfg.seed,
        min_df=baseline_cfg.min_df,
        test_size=baseline_cfg.test_size,
        cv_splits=baseline_cfg.cv_splits,
        include_bigrams=baseline_cfg.include_bigrams,
        use_smote=baseline_cfg.use_smote,
        use_corpus_stats_features=True,
        corpus_stats_feature_columns=None
        if selected_features is None
        else tuple(selected_features),  # uses selected features
        mlp_kwargs=dict(baseline_cfg.mlp_kwargs),
    )


def run_configuration(
    configuration_name: str,
    selected_features: list[str] | tuple[str, ...] | None,
    removed_feature: str | None,
) -> tuple[dict[str, object], object]:
    """Runs an MLP model using the baseline configs and the selected CorpusStats features"""
    cfg = make_config(selected_features)
    results = run_mlp_authorship_pipeline(
        train_data=train_texts,
        train_labels=train_labels,
        test_data=unknown_texts,
        test_ids=unknown_ids,
        config=cfg,
    )

    row = {
        "configuration": configuration_name,
        "removed_feature": removed_feature or "baseline",
        "features_remaining": len(all_corpus_stat_features)
        if selected_features is None
        else len(selected_features),
        "holdout_accuracy": results.holdout_metrics["accuracy"],
        "holdout_balanced_accuracy": results.holdout_metrics["balanced_accuracy"],
        "holdout_macro_f1": results.holdout_metrics["macro_f1"],
        "cv_accuracy": results.cv_mean_metrics["accuracy"],
        "cv_balanced_accuracy": results.cv_mean_metrics["balanced_accuracy"],
        "cv_macro_f1": results.cv_mean_metrics["macro_f1"],
        "predicted_hamilton": int(
            (results.test_predictions["predicted_label"] == "HAMILTON").sum()
        )
        if not results.test_predictions.empty
        else 0,
        "predicted_madison": int(
            (results.test_predictions["predicted_label"] == "MADISON").sum()
        )
        if not results.test_predictions.empty
        else 0,
    }
    return row, results


experiment_rows: list[dict[str, object]] = []
configuration_outputs: dict[str, object] = {}

baseline_row, baseline_results = run_configuration(
    "baseline", all_corpus_stat_features, None
)
experiment_rows.append(baseline_row)
configuration_outputs["baseline"] = baseline_results

active_features = list(all_corpus_stat_features)
for step_number, removed_feature in enumerate(feature_removal_order, start=1):
    active_features = [
        feature for feature in active_features if feature != removed_feature
    ]
    configuration_name = f"remove_{step_number:02d}"
    print(
        f"Running {configuration_name}: removed {removed_feature}; {len(active_features)} features remain"
    )
    row, results = run_configuration(
        configuration_name, active_features, removed_feature
    )
    experiment_rows.append(row)
    configuration_outputs[configuration_name] = results

results_df = pd.DataFrame(experiment_rows)
baseline_metrics = results_df.loc[results_df["configuration"] == "baseline"].iloc[0]

for metric_name in [
    "holdout_accuracy",
    "holdout_balanced_accuracy",
    "holdout_macro_f1",
    "cv_accuracy",
    "cv_balanced_accuracy",
    "cv_macro_f1",
]:
    results_df[f"{metric_name}_delta_vs_baseline"] = results_df[metric_name] - float(
        baseline_metrics[metric_name]
    )

summary_columns = [
    "configuration",
    "removed_feature",
    "features_remaining",
    "holdout_accuracy",
    "holdout_balanced_accuracy",
    "holdout_macro_f1",
    "cv_accuracy",
    "cv_balanced_accuracy",
    "cv_macro_f1",
    "holdout_macro_f1_delta_vs_baseline",
]

print()
print("Feature removal sweep:")
print(results_df[summary_columns].to_string(index=False))

impact_row = results_df.loc[results_df["holdout_macro_f1_delta_vs_baseline"].idxmin()]
print()
print("Greatest holdout macro-F1 drop vs baseline:")
print(
    f"{impact_row['removed_feature']} | "
    f"delta={impact_row['holdout_macro_f1_delta_vs_baseline']:.4f} | "
    f"holdout_macro_f1={impact_row['holdout_macro_f1']:.4f}"
)

best_row = results_df.loc[results_df["holdout_macro_f1"].idxmax()]
print()
print("Best holdout macro-F1 configuration:")
print(
    f"{best_row['configuration']} | "
    f"removed={best_row['removed_feature']} | "
    f"features_remaining={int(best_row['features_remaining'])} | "
    f"holdout_macro_f1={best_row['holdout_macro_f1']:.4f}"
)

## 4.5 Understanding the Performance Metrics

The experiment measures six key metrics across two evaluation contexts to comprehensively assess model performance:

### Evaluation Contexts

**Holdout Metrics**: Evaluated on the test set (20% of training data) that was never seen during model training. These represent real-world generalization performance.

**Cross-Validation (CV) Metrics**: Averaged across 5 folds during training. These provide a more robust estimate of performance by testing on multiple train/test splits.

### The Six Metrics

**Accuracy**: The percentage of documents correctly classified overall. While simple to interpret, it can be misleading with imbalanced classes (e.g., if one author has more papers, a model could achieve high accuracy by always predicting that author).

**Balanced Accuracy**: The average of per-class recall scores. This is crucial for authorship attribution because it accounts for class imbalance and ensures both Hamilton and Madison are equally well-predicted. A model with high accuracy but low balanced accuracy is likely biased toward one author.

**Macro F1**: The average F1 score across both classes. F1 balances precision (how many predicted Hamilton papers are actually Hamilton) and recall (how many actual Hamilton papers are correctly identified). Macro F1 treats both authors equally, making it ideal for this balanced binary classification task.
